# Brahmacharini: which ATT&CK technique is this behaviour?

Real attacker behaviours from MITRE ATT&CK, each labelled with its technique. The test split holds groups, malware and tools never seen in training. Below: a TF-IDF retriever, a trained classifier, and what a random split would have claimed.

Brahmacharini is one of the nine models of **Sati**, the investigation agent of [Pashupatastra](https://github.com/preetraval45/PASHUPATASTRA), each named for a form of Navadurga. This notebook reproduces its core result on the public dataset, with the split that keeps the test honest.

In [ ]:
import glob, os
def find(pattern):
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    assert hits, f"{pattern} not found under /kaggle/input"
    return hits[0]
import pandas as pd, numpy as np
b = pd.read_csv(find("behaviours.csv")); t = pd.read_csv(find("techniques.csv"))
print(b["split"].value_counts().to_dict(), len(t), "techniques")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
train, test = b[b["split"] == "train"], b[b["split"] == "test"]
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit(pd.concat([train["behaviour"], t["text"]]))
sims = (vec.transform(test["behaviour"]) @ vec.transform(t["text"]).T).toarray()
ranked = t["technique_id"].values[np.argsort(-sims, axis=1)]
print(f"TF-IDF retrieval  recall@1 {np.mean(ranked[:, 0] == test['technique_id'].values):.3f}  "
      f"recall@10 {np.mean([g in r[:10] for g, r in zip(test['technique_id'], ranked)]):.3f}")
clf = LogisticRegression(max_iter=2000, C=5).fit(vec.transform(train["behaviour"]), train["technique_id"])
print(f"trained classifier recall@1 {clf.score(vec.transform(test['behaviour']), test['technique_id']):.3f}")

In [ ]:
from sklearn.model_selection import train_test_split
ra, rb = train_test_split(b, test_size=0.2, random_state=7)
leaky = LogisticRegression(max_iter=2000, C=5).fit(vec.transform(ra["behaviour"]), ra["technique_id"])
print(f"same classifier on a random split: {leaky.score(vec.transform(rb['behaviour']), rb['technique_id']):.3f}")

**Reading it:** two honest surprises. First, the random split scores the same as the by-source split here, so for this classifier, a group's phrasing does not leak much. Second, this simple TF-IDF + logistic regression classifier (about 0.65 recall@1) **beats Brahmacharini** (0.614), the project's dual encoder, on the same split. Brahmacharini's advantage is elsewhere: one vector space for behaviours, techniques and knowledge-base passages, so it also retrieves the passage Sati cites. The project now reports this baseline beside it.